# Étape 17a — Deuxième jeu de données : exploration de DBE-KT22

**Objectif de l'étape 17.** Toutes nos conclusions viennent pour l'instant de **Junyi** (mathématiques, Taïwan), où les élèves suivent l'ordre de la carte des experts.
Un relecteur demandera : *« est-ce vrai ailleurs ? »*. On refait donc l'essentiel de l'étude sur un **deuxième jeu de données**, très différent :
**DBE-KT22** (Abdelrahman et al., 2022) — un cours universitaire de **bases de données** (Australian National University), 1 361 étudiants, 212 questions, 98 notions (KC), environ 167 000 réponses,
et surtout un fichier de **relations entre notions** (`KC_Relationships.csv`) qui sert de graphe expert.

**Ce notebook (17a)** ne fait aucun entraînement. Il vérifie que les données permettent l'étude :
1. que les relations entre notions sont bien des **prérequis** (on affiche des exemples avec les noms des notions) ;
2. combien de questions ont au moins un prérequis (**couverture**) ;
3. la taille des séquences d'élèves et la part de **premières tentatives**.

D'après le résultat, je préparerai le notebook 17b (préparation des données au même format que Junyi, puis l'expérience couverture × justesse).

### Avant de lancer : télécharger DBE-KT22 (une seule fois)
1. Ouvrez https://dataverse.ada.edu.au/dataset.xhtml?persistentId=doi:10.26193/6DZWOH dans votre navigateur.
2. Cliquez sur **« Access Dataset » → « Download ZIP »** (acceptez les conditions si on vous les demande ; si un compte est demandé, créez-le, c'est gratuit).
3. Dans votre Drive, créez le dossier `projet_kt/dbe_kt22` et déposez-y le **zip** (ou les fichiers `.csv` déjà extraits).

**Pas besoin de GPU.** Durée : 1 à 2 minutes.

In [ ]:
# Cellule 1 — Drive et décompression
from google.colab import drive
drive.mount('/content/drive')

import os, glob, zipfile, json, re
import numpy as np
import pandas as pd
import networkx as nx

DOSSIER   = '/content/drive/MyDrive/projet_kt'
DBE       = os.path.join(DOSSIER, 'dbe_kt22')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)

for z in glob.glob(os.path.join(DBE, '*.zip')):
    with zipfile.ZipFile(z) as f:
        f.extractall(DBE)
    print('décompressé :', os.path.basename(z))
# certains zips Dataverse contiennent d'autres zips
for z in glob.glob(os.path.join(DBE, '**', '*.zip'), recursive=True):
    with zipfile.ZipFile(z) as f:
        f.extractall(os.path.dirname(z))

CSV = sorted(glob.glob(os.path.join(DBE, '**', '*.csv'), recursive=True))
print(f'{len(CSV)} fichiers csv trouvés :')
for c in CSV:
    print(f'  {os.path.relpath(c, DBE):60s} {os.path.getsize(c) / 1e6:7.2f} Mo')

In [ ]:
# Cellule 2 — Colonnes et premières lignes de chaque fichier
TABLES = {}
for c in CSV:
    nom = os.path.splitext(os.path.basename(c))[0]
    try:
        df = pd.read_csv(c, low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(c, low_memory=False, encoding='latin-1')
    TABLES[nom] = df
    print(f'\n===== {nom} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
    print('colonnes :', list(df.columns))
    with pd.option_context('display.max_columns', 20, 'display.width', 200, 'display.max_colwidth', 40):
        print(df.head(3).to_string())

In [ ]:
# Cellule 3 — Repérage automatique des tables utiles
def table(*mots):
    for nom, df in TABLES.items():
        n = nom.lower().replace('-', '_')
        if all(m in n for m in mots):
            return nom, df
    return None, None

def colonne(df, *motifs, exclure=()):
    for c in df.columns:
        cl = c.lower()
        if any(m in cl for m in motifs) and not any(e in cl for e in exclure):
            return c
    return None

n_tr, TR = table('transaction')
n_kr, KR = table('kc', 'relationship')
if KR is None or 'question' in (n_kr or '').lower():
    for nom, df in TABLES.items():
        if 'kc' in nom.lower() and 'relationship' in nom.lower() and 'question' not in nom.lower():
            n_kr, KR = nom, df
n_qk, QK = table('question', 'kc')
n_kc, KC = None, None
for nom, df in TABLES.items():
    if nom.lower() in ('kcs', 'kc', 'knowledge_components'):
        n_kc, KC = nom, df
n_q, QS = None, None
for nom, df in TABLES.items():
    if nom.lower() in ('questions', 'question'):
        n_q, QS = nom, df
print('transactions   :', n_tr)
print('relations KC   :', n_kr)
print('question ↔ KC  :', n_qk)
print('notions (KC)   :', n_kc)
print('questions      :', n_q)

In [ ]:
# Cellule 4 — Les réponses des étudiants
resume = {}
c_u = colonne(TR, 'student')
c_q = colonne(TR, 'question')
c_r = colonne(TR, 'answer_state', 'correct', 'is_right', 'score')
c_t = colonne(TR, 'start_time', 'time', 'date', exclure=('end',))
print('colonnes utilisées : élève =', c_u, '| question =', c_q, '| réussite =', c_r, '| temps =', c_t)
T = TR[[c_u, c_q, c_r] + ([c_t] if c_t else [])].dropna(subset=[c_u, c_q, c_r]).copy()
T.columns = ['u', 'q', 'r'] + (['t'] if c_t else [])
print('valeurs de réussite :', T['r'].value_counts().head(6).to_dict())
T['r'] = T['r'].astype(str).str.lower().isin(['1', '1.0', 'true', 't', 'yes', 'correct'])
if c_t:
    T['t'] = pd.to_datetime(T['t'], errors='coerce')
    T = T.sort_values(['u', 't'], kind='stable')
T['premiere'] = ~T.duplicated(['u', 'q'])
long = T.groupby('u').size()
resume['reponses'] = {'etudiants': int(T.u.nunique()), 'questions_repondues': int(T.q.nunique()), 'reponses': int(len(T)),
                      'taux_reussite': round(float(T.r.mean()), 3), 'part_premieres_tentatives': round(float(T.premiere.mean()), 3),
                      'longueur_sequence_mediane': int(long.median()), 'longueur_sequence_max': int(long.max()),
                      'etudiants_10_reponses_ou_plus': int((long >= 10).sum())}
print(json.dumps(resume['reponses'], indent=1, ensure_ascii=False))

In [ ]:
# Cellule 5 — Les relations entre notions : est-ce un graphe de prérequis ?
cols_kr = list(KR.columns)
c_de = colonne(KR, 'from', 'source', 'prereq', 'parent', 'kc_id_1', 'knowledgecomponent_id', exclure=('to_',))
c_vers = colonne(KR, 'to_', 'target', 'dependent', 'child', 'kc_id_2', exclure=(c_de.lower(),) if c_de else ())
print('colonnes :', cols_kr, '\n=> de =', c_de, '| vers =', c_vers)
autres = [c for c in cols_kr if c not in (c_de, c_vers)]
for c in autres:
    print(f'colonne « {c} » :', KR[c].astype(str).value_counts().head(8).to_dict())

# noms des notions
c_kid = colonne(KC, 'id') if KC is not None else None
c_knom = colonne(KC, 'name', 'title', 'description') if KC is not None else None
nom_kc = dict(zip(KC[c_kid], KC[c_knom])) if c_kid and c_knom else {}
E = list(zip(KR[c_de], KR[c_vers]))
G = nx.DiGraph(); G.add_edges_from(E)
print(f'\n{len(E)} relations | {G.number_of_nodes()} notions concernées (sur {len(KC) if KC is not None else "?"}) | '
      f'acyclique : {nx.is_directed_acyclic_graph(G)} | paires dans les deux sens : {sum(1 for a, b in E if (b, a) in set(E))}')
print('\n20 exemples (de → vers) — LISEZ-LES : la notion de gauche doit être nécessaire pour apprendre celle de droite')
for a, b in E[:20]:
    print(f'  {str(nom_kc.get(a, a))[:45]:45s} → {str(nom_kc.get(b, b))[:45]}')
deg_in = pd.Series(dict(G.in_degree())); deg_out = pd.Series(dict(G.out_degree()))
print('\nNotions avec le plus de flèches sortantes (souvent les bases) :')
for k in deg_out.sort_values(ascending=False).head(5).index:
    print(f'  {str(nom_kc.get(k, k))[:50]:50s} sortantes = {deg_out[k]} | entrantes = {deg_in[k]}')
resume['relations_kc'] = {'colonnes': cols_kr, 'de': c_de, 'vers': c_vers, 'relations': len(E),
                          'notions_concernees': G.number_of_nodes(), 'notions_total': int(len(KC)) if KC is not None else None,
                          'acyclique': bool(nx.is_directed_acyclic_graph(G)),
                          'exemples': [[str(nom_kc.get(a, a)), str(nom_kc.get(b, b))] for a, b in E[:15]]}

In [ ]:
# Cellule 6 — Des notions aux questions : graphe de prérequis entre questions et couverture
c_qq = colonne(QK, 'question')
c_qk = colonne(QK, 'knowledgecomponent', 'kc')
QK_ = QK[[c_qq, c_qk]].dropna()
kc_par_q = QK_.groupby(c_qq)[c_qk].apply(set)
q_par_kc = QK_.groupby(c_qk)[c_qq].apply(set)
print('notions par question :', kc_par_q.apply(len).value_counts().sort_index().to_dict())
print('questions par notion : médiane =', int(q_par_kc.apply(len).median()), '| max =', int(q_par_kc.apply(len).max()))

# Question A -> question B si une notion de A est prérequis (direct) d'une notion de B, et A, B n'ont aucune notion commune
EQ = set()
for a_kc, b_kc in E:
    for qa in q_par_kc.get(a_kc, ()):
        for qb in q_par_kc.get(b_kc, ()):
            if qa != qb and not (kc_par_q[qa] & kc_par_q[qb]):
                EQ.add((qa, qb))
questions = sorted(set(T.q))
avec_prereq = {b for _, b in EQ}
prem = T[T.premiere]
couv_q = np.mean([q in avec_prereq for q in questions])
couv_1res = prem.q.isin(avec_prereq).mean()
# notion « thème » : questions qui partagent au moins une notion
meme_notion = sum(1 for a, b in EQ if kc_par_q[a] & kc_par_q[b])
resume['questions'] = {'fleches_entre_questions': len(EQ), 'questions': len(questions),
                       'couverture_questions_%': round(100 * couv_q, 1), 'couverture_1res_tentatives_%': round(100 * couv_1res, 1),
                       'prerequis_par_question_couverte': round(len(EQ) / max(1, len(avec_prereq)), 1)}
print(json.dumps(resume['questions'], indent=1, ensure_ascii=False))

In [ ]:
# Cellule 7 — Résumé à m'envoyer (copiez tout ce qui s'affiche, ou envoyez le fichier dbe_kt22_exploration.json)
chemin = os.path.join(RESULTATS, 'dbe_kt22_exploration.json')
json.dump(resume, open(chemin, 'w'), indent=1, ensure_ascii=False, default=str)
print(json.dumps(resume, indent=1, ensure_ascii=False, default=str))
print('\nEnregistré :', chemin)